# Session 23: Project 3: Your First ML Model

Data Science with Python, Level 1, Stage 3: Statistics and ML Fundamentals

## Learning Objectives

By the end of this session you will be able to:

1. Frame a prediction problem as a business decision.
2. Clean and encode a raw table for a model.
3. Compare candidate models with cross-validation.
4. Pick a threshold that matches the business goal.
5. Measure the final model once on the test set.
6. Explain the model in a short report.

## Where We Are

Level 1 ends with your third portfolio project. In Stage 3 you learned to describe data, to test whether a difference is real, and to build, score and cross-validate your first models. Today you put all of it together into one clean, honest machine learning project.

The deliverable is a notebook that a hiring manager or a retention team could read: what problem it solves, how good the model really is, how to use it, and what drives the predictions. You may use the churn data or another course dataset with your instructor's approval.

## Setup

Run this cell first. It loads the churn data and the target: 1 means the customer left.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.metrics import roc_auc_score, recall_score, precision_score, confusion_matrix

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
y = (churn["churn"] == "Yes").astype(int)

## 1. The Project Brief

| Dataset | Target | Decision it supports |
| --- | --- | --- |
| `telecom_churn.csv` | churn | Which customers should the retention team call? |
| `card_transactions.csv` | is_fraud | Which payments should be checked? |
| `apartments.csv` | price_k | What is a fair asking price for a flat? |
| `exam_scores.csv` | passed | Which students need extra support early? |

Regression projects replace the threshold milestone with an error analysis: where is the model most wrong?

## 2. The Modelling Workflow

| Step | What you do |
| --- | --- |
| 1. Frame | Target, decision, metric and the cost of each error |
| 2. Prepare | Fill missing values, turn text columns into numbers |
| 3. Split | Set aside the test set, stratified, and do not touch it |
| 4. Baseline | Majority class and a one-feature model |
| 5. Compare | Candidate models with cross-validation |
| 6. Threshold | Choose it from cross-validated predictions |
| 7. Test once | Fit on all development data, score the test set |
| 8. Explain | Odds ratios and a written recommendation |

In [ ]:
def prepare(df):
    out = df.drop(columns=["customer_id", "churn"]).copy()
    out["internet"] = out["internet"].fillna("None")
    out["total_charges"] = out["total_charges"].fillna(out["tenure_months"] * out["monthly_charges"])
    out["male"] = (out.pop("gender") == "Male").astype(int)
    out = pd.get_dummies(out, columns=["contract", "internet", "payment_method"], drop_first=True)
    return out.astype(float)

X = prepare(churn)
print(X.shape)
print(X.columns.tolist())

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2,
                                                stratify=y, random_state=0)
print("Majority accuracy:", round(1 - y_dev.mean(), 3))

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
for name, cols in [("tenure only", ["tenure_months"]),
                   ("three numbers", ["tenure_months", "monthly_charges", "support_calls"]),
                   ("all features", list(X.columns))]:
    scores = cross_val_score(model, X_dev[cols], y_dev, cv=5, scoring="roc_auc")
    print(f"{name:14} AUC {scores.mean():.3f} +/- {scores.std():.3f}")

## 3. Choosing the Threshold

`cross_val_predict` works like `cross_val_score`, but instead of a score per fold it returns a **prediction for every development row**, each made by a model that did not see that row. These honest probabilities let you choose a threshold without using the test set.

In [ ]:
oof = cross_val_predict(model, X_dev, y_dev, cv=5,
                        method="predict_proba")[:, 1]

for t in [0.2, 0.3, 0.4, 0.5]:
    p = (oof >= t).astype(int)
    print(t, "recall", round(recall_score(y_dev, p), 2),
          "precision", round(precision_score(y_dev, p), 2))

In [ ]:
model.fit(X_dev, y_dev)
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.3).astype(int)

print("Test AUC:", round(roc_auc_score(y_test, test_proba), 3))
print("Recall:", round(recall_score(y_test, test_pred), 2),
      " Precision:", round(precision_score(y_test, test_pred), 2))
print(confusion_matrix(y_test, test_pred))

In [ ]:
odds = pd.Series(np.exp(model[-1].coef_[0]), index=X.columns).sort_values()
print(odds.round(2).iloc[[0, 1, 2, -3, -2, -1]])

| Item | Content |
| --- | --- |
| Purpose | Rank customers by risk of leaving in the next period |
| Data | 2,000 practice customers, 10 input columns, 30 percent churn |
| Model | Logistic regression with scaling, in a scikit-learn pipeline |
| Performance | Test AUC 0.80; at threshold 0.3: recall 0.78, precision 0.53 |
| Main drivers | Monthly contract, short tenure, high charges, support calls |
| Limits | Practice data; associations, not causes; retrain as offers change |

## 4. How the Project Is Graded

| Criterion | Points | What earns full points |
| --- | --- | --- |
| Framing | 10 | Clear target, decision, metric and error costs |
| Data preparation | 20 | Missing values handled with reasons, text columns encoded, no leakage |
| Model comparison | 20 | Baselines and at least two candidates compared with cross-validation |
| Evaluation | 20 | Threshold from cross-validation, test set used once, right metrics |
| Explanation | 20 | Coefficients or odds ratios, plain-language findings |
| Reproducible | 10 | Runs top to bottom; random seeds set |

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Peeking | Checking the test score after every change | Use the test set once, at the very end. |
| No baseline | Reporting AUC 0.80 without context | Compare with a majority-class or one-feature model. |
| Threshold 0.5 by habit | Ignoring what the business needs | Choose the threshold from the goal and costs. |
| Jargon | Writing "AUC improved 0.004" for managers | Translate: "we find 94 of 121 leavers". |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Frame the Problem

Write the framing for your project in a text cell: target, positive class, decision, main metric, and the cost of a false positive and a false negative. Print it from a code cell.

In [ ]:
# Write your code below

### In-Class Exercise 2: Missing Values

Count the missing values in each column of `churn`. For each column with gaps, look at a few of those rows and write one sentence on why the value is missing and how you will fill it.

In [ ]:
# Write your code below

### In-Class Exercise 3: Call List

Using the final model, list the 10 test customers with the highest predicted risk, with their contract, tenure, charges and whether they really left. How many of the 10 did leave?

In [ ]:
# Write your code below

### In-Class Exercise 4: Peer Review

Swap notebooks with a classmate. Using the rubric, write two strengths and two improvements as comments, and check one thing carefully: is the test set used only once?

In [ ]:
# Write your code below

## Summary

- Frame the problem as a decision with costs.
- Prepare the table in one function: fill gaps, encode text columns.
- Split off the test set and keep it untouched.
- Compare candidates and baselines with cross-validation.
- Choose the threshold from cross-validated predictions; score the test set once.
- A model card states purpose, performance and limits.

## What's Next

Level 1 ends here. Level 2 starts with Stage 4, Applied ML: today you prepared the table by hand; next session scikit-learn transformers do it inside one pipeline.